# SpaceX Falcon 9 SQL Analysis


This notebook analyzes SpaceX Falcon 9 launch data to explore factors affecting first-stage landing success and build predictive models.


## Objective
- Load the structured launch dataset into SQLite.
- Answer key business and mission queries with SQL.
- Keep the notebook runnable in a standard Python environment without notebook magics.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / "data").exists() and (candidate / "notebooks").exists()
)
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOKS_DIR))

import sqlite3

import pandas as pd

from notebook_utils import data_path, ensure_binary_file


## Data Loading


In [2]:
DATASET_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"
)
dataset_path = data_path("Spacex.csv")
if not dataset_path.exists():
    dataset_path = ensure_binary_file("Spacex.csv", DATASET_URL)

df = pd.read_csv(dataset_path)
df.head()


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


## Data Cleaning


In [3]:
db_path = data_path("spacex.sqlite")
con = sqlite3.connect(db_path)
clean_df = df[df["Date"].notna()].copy()
clean_df.to_sql("SPACEXTBL", con, if_exists="replace", index=False)


def run_query(query: str) -> pd.DataFrame:
    return pd.read_sql_query(query, con)


## Exploratory Data Analysis


In [4]:
queries = {
    "distinct_launch_sites": "SELECT DISTINCT LAUNCH_SITE FROM SPACEXTBL;",
    "cca_launches": "SELECT * FROM SPACEXTBL WHERE LAUNCH_SITE LIKE 'CCA%' LIMIT 5;",
    "nasa_crs_payload_mass": "SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_mass FROM SPACEXTBL WHERE Customer = 'NASA (CRS)';",
    "average_payload_mass": "SELECT AVG(PAYLOAD_MASS__KG_) AS average_payload_mass FROM SPACEXTBL;",
    "first_ground_pad_success": "SELECT MIN(DATE) AS first_ground_pad_success FROM SPACEXTBL WHERE LANDING_OUTCOME = 'Success (ground pad)';",
    "drone_ship_boosters": "SELECT BOOSTER_VERSION FROM SPACEXTBL WHERE LANDING_OUTCOME = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000;",
    "mission_outcome_counts": "SELECT MISSION_OUTCOME, COUNT(*) AS mission_outcomes FROM SPACEXTBL GROUP BY MISSION_OUTCOME;",
    "heaviest_payload_booster": "SELECT BOOSTER_VERSION AS booster_version FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL);",
    "launches_in_2015": "SELECT SUBSTR(Date, 6, 2) AS month, MISSION_OUTCOME, BOOSTER_VERSION, LAUNCH_SITE FROM SPACEXTBL WHERE SUBSTR(Date, 1, 4) = '2015';",
    "landing_outcomes_between_dates": "SELECT COUNT(LANDING_OUTCOME) AS landing_outcomes FROM SPACEXTBL WHERE DATE BETWEEN '2010-06-04' AND '2017-03-20';",
}

results = {name: run_query(query) for name, query in queries.items()}
results["distinct_launch_sites"]


,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


In [5]:
results["cca_launches"]


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [6]:
results["nasa_crs_payload_mass"]


,total_payload_mass
0,45596


In [7]:
results["average_payload_mass"]


,average_payload_mass
0,6138.287129


In [8]:
results["first_ground_pad_success"]


,first_ground_pad_success
0,2015-12-22


In [9]:
results["drone_ship_boosters"]


,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


In [10]:
results["mission_outcome_counts"]


,Mission_Outcome,mission_outcomes
0,Failure (in flight),1
1,Success,98
2,Success,1
3,Success (payload status unclear),1


In [11]:
results["heaviest_payload_booster"]


,booster_version
0,F9 B5 B1048.4
1,F9 B5 B1049.4
2,F9 B5 B1051.3
3,F9 B5 B1056.4
4,F9 B5 B1048.5
5,F9 B5 B1051.4
6,F9 B5 B1049.5
7,F9 B5 B1060.2
8,F9 B5 B1058.3
9,F9 B5 B1051.6


In [12]:
results["launches_in_2015"]


,month,Mission_Outcome,Booster_Version,Launch_Site
0,01,Success,F9 v1.1 B1012,CCAFS LC-40
1,02,Success,F9 v1.1 B1013,CCAFS LC-40
2,03,Success,F9 v1.1 B1014,CCAFS LC-40
3,04,Success,F9 v1.1 B1015,CCAFS LC-40
4,04,Success,F9 v1.1 B1016,CCAFS LC-40
5,06,Failure (in flight),F9 v1.1 B1018,CCAFS LC-40
6,12,Success,F9 FT B1019,CCAFS LC-40


In [13]:
results["landing_outcomes_between_dates"]


,landing_outcomes
0,31


## Key Findings
These SQL checks highlight launch-site coverage, payload trends, and mission outcomes directly from the structured SpaceX launch table.
